# 📊 Etapa 3 — Avaliação dos 4 Modelos Fine-Tuned com LoRA

Avaliamos os 4 modelos treinados usando as métricas exigidas pelas diretrizes:
PPL, BLEU, ROUGE-1/2/L, Fidelidade, Relevância (Jaccard) e Aderência ao Plano.


In [ ]:
!pip uninstall -y gradio
!pip install "pillow<12.0.0" "peft==0.10.0" "datasets==2.18.0" "sacrebleu==2.4.0" "rouge-score==0.1.2" transformers accelerate torch nltk pandas matplotlib seaborn -q


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Dependências instaladas!')

In [ ]:
import json, math, re, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Dispositivo: {DEVICE}')


## 1. Carregamento do Dataset

Usamos o `dataset_gerado.jsonl` com chaves `Instruction` e `Output`.


In [ ]:
samples = []
with open('dataset_gerado.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f'✅ Dataset: {len(samples)} amostras')
print(f'Chaves: {list(samples[0].keys())}')
print(f'\nExemplo [0]:')
print(f'  Instruction: {samples[0]["Instruction"]}')
print(f'  Output: {samples[0]["Output"]}')


## 2. Configuração e Carregamento dos 4 Modelos


In [ ]:
MODELOS = [
    {'nome': 'EleutherAI/pythia-160m', 'path': 'lora_causal_pythia_160m',
     'tok_path': 'tokenizer_pythia_160m', 'tipo': 'causal'},
    {'nome': 'EleutherAI/gpt-neo-125m', 'path': 'lora_causal_gpt_neo_125m',
     'tok_path': 'tokenizer_gpt_neo_125m', 'tipo': 'causal'},
    {'nome': 'google/flan-t5-base', 'path': 'lora_seq2seq_flan_t5_base',
     'tok_path': 'tokenizer_flan_t5_base', 'tipo': 'seq2seq'},
    {'nome': 'google/mt5-small', 'path': 'lora_seq2seq_mt5_small',
     'tok_path': 'tokenizer_mt5_small', 'tipo': 'seq2seq'}
]

loaded_models = {}
for m in MODELOS:
    print(f'⏳ Carregando {m["nome"]}...')
    try:
        tok = AutoTokenizer.from_pretrained(m['tok_path'])
        if m['tipo'] == 'causal' and tok.pad_token is None:
            tok.pad_token = tok.eos_token
        if m['tipo'] == 'causal':
            base = AutoModelForCausalLM.from_pretrained(m['nome'])
        else:
            base = AutoModelForSeq2SeqLM.from_pretrained(m['nome'])
        model = PeftModel.from_pretrained(base, m['path'])
        model.eval().to(DEVICE)
        loaded_models[m['nome']] = {'model': model, 'tokenizer': tok, 'tipo': m['tipo']}
        print(f'  ✅ {m["nome"]} carregado!')
    except Exception as e:
        print(f'  ❌ Erro: {e}')


## 3. Funções de Geração


In [ ]:
def generate_response(m_dict, instruction, max_new_tokens=100):
    tok = m_dict['tokenizer']
    mod = m_dict['model']
    tipo = m_dict['tipo']
    prompt = f'Instruction: {instruction}\nOutput:'
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        if tipo == 'causal':
            out = mod.generate(**inputs, max_new_tokens=max_new_tokens,
                               do_sample=True, temperature=0.7,
                               pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
            full = tok.decode(out[0], skip_special_tokens=True)
            return full.split('Output:')[-1].strip()
        else:
            inputs2 = tok(f'Instruction: {instruction}', return_tensors='pt',
                          truncation=True, max_length=128).to(DEVICE)
            out = mod.generate(**inputs2, max_new_tokens=max_new_tokens)
            return tok.decode(out[0], skip_special_tokens=True).strip()

print('✅ Funções de geração definidas!')


## 4. Geração das Respostas e Cálculo de Métricas

Calculamos PPL, BLEU, ROUGE-1/2/L, Faithfulness (sobreposição léxica), Jaccard (Answer Relevance) e Plan Adherence para os 4 modelos.


In [ ]:
def compute_ppl(model_dict, samples):
    mod = model_dict['model']
    tok = model_dict['tokenizer']
    tipo = model_dict['tipo']
    ppls = []
    for s in samples:
        if tipo != 'causal': ppls.append(float('nan')); continue
        text = f"Instruction: {s['Instruction']}\nOutput: {s['Output']}"
        enc = tok(text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
        labels = enc['input_ids'].clone()
        # Mask prompt tokens
        prompt_len = len(tok(f"Instruction: {s['Instruction']}\nOutput:")['input_ids'])
        labels[0, :prompt_len] = -100
        with torch.no_grad():
            out = mod(**enc, labels=labels)
        if not torch.isnan(out.loss):
            ppls.append(math.exp(min(out.loss.item(), 20)))
    return np.nanmean(ppls) if ppls else float('nan')

def compute_bleu(hyps, refs):
    bleu = BLEU(effective_order=True)
    res = bleu.corpus_score(hyps, [refs])
    return res.score

def compute_rouge(hyps, refs):
    sc = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for h, r in zip(hyps, refs):
        s = sc.score(r, h)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return np.mean(r1), np.mean(r2), np.mean(rl)

def compute_faithfulness(samples, hyps):
    scores = []
    for s, h in zip(samples, hyps):
        ctx = set(word_tokenize(s['Instruction'].lower()))
        gen = set(word_tokenize(h.lower())) if h.strip() else set()
        scores.append(len(ctx & gen) / len(gen) if gen else 0)
    return np.mean(scores)

def compute_jaccard(samples, hyps):
    scores = []
    for s, h in zip(samples, hyps):
        a = set(word_tokenize(s['Instruction'].lower()))
        b = set(word_tokenize(h.lower())) if h.strip() else set()
        scores.append(len(a & b) / len(a | b) if (a | b) else 0)
    return np.mean(scores)

def compute_plan_adherence(samples, hyps):
    """Verifica se resposta segue estrutura: listas, valores técnicos."""
    scores = []
    for s, h in zip(samples, hyps):
        ref = s['Output']
        has_num_ref = bool(re.search(r'\d', ref))
        has_num_hyp = bool(re.search(r'\d', h))
        has_list_ref = bool(re.search(r'(\d\.|-\s|•)', ref))
        has_list_hyp = bool(re.search(r'(\d\.|-\s|•)', h))
        score = 0
        if has_num_ref == has_num_hyp: score += 0.5
        if has_list_ref == has_list_hyp: score += 0.5
        scores.append(score)
    return np.mean(scores)

# Avaliação
resultados = []
respostas_por_modelo = {}

for name, m_dict in loaded_models.items():
    print(f"\n{'='*50}\n📊 Avaliando: {name}")
    hyps, refs = [], []
    for s in tqdm(samples, desc=name.split('/')[-1]):
        hyps.append(generate_response(m_dict, s['Instruction']))
        refs.append(s['Output'])

    respostas_por_modelo[name] = hyps
    ppl = compute_ppl(m_dict, samples)
    bleu = compute_bleu(hyps, refs)
    r1, r2, rl = compute_rouge(hyps, refs)
    faith = compute_faithfulness(samples, hyps)
    jaccard = compute_jaccard(samples, hyps)
    plan = compute_plan_adherence(samples, hyps)

    resultados.append({
        'Modelo': name.split('/')[-1], 'PPL': round(ppl, 2),
        'BLEU': round(bleu, 2), 'ROUGE-1': round(r1, 3),
        'ROUGE-2': round(r2, 3), 'ROUGE-L': round(rl, 3),
        'Faithfulness': round(faith, 3), 'Jaccard': round(jaccard, 3),
        'Plan Adherence': round(plan, 3)
    })
    print(f'  PPL={ppl:.1f} | BLEU={bleu:.2f} | R-L={rl:.3f} | Faith={faith:.3f} | Jaccard={jaccard:.3f}')


## 5. Tabela Comparativa


In [ ]:
df = pd.DataFrame(resultados)
df.to_csv('resultados_avaliacao.csv', index=False)
print('📊 Tabela Comparativa de Métricas:')
print(df.to_string(index=False))
print(f'\nMelhor modelo (ROUGE-L): {df.loc[df["ROUGE-L"].idxmax(), "Modelo"]}')


## 6. Gráfico Radar


In [ ]:
metricas_radar = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Faithfulness', 'Jaccard', 'Plan Adherence']
N = len(metricas_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
cores = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

for i, row in df.iterrows():
    vals = [row[m] / (df[m].max() if df[m].max() > 0 else 1) for m in metricas_radar]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=row['Modelo'], color=cores[i % 4])
    ax.fill(angles, vals, alpha=0.1, color=cores[i % 4])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metricas_radar, fontsize=10)
ax.set_title('Comparação dos 4 Modelos LoRA', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('avaliacao_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Gráfico radar salvo: avaliacao_metricas.png')


## 7. Análise Qualitativa (3 Amostras)

Comparamos a resposta do modelo base, do modelo fine-tunado e a resposta de referência.


In [ ]:
AMOSTRAS_QUALITATIVAS = [0, 1, 2]

for idx in AMOSTRAS_QUALITATIVAS:
    s = samples[idx]
    print(f"\n{'='*60}")
    print(f'📌 Amostra #{idx+1}')
    print(f'Instrução: {s["Instruction"]}')
    print(f'\n✅ Referência: {s["Output"]}')
    for name, hyps in respostas_por_modelo.items():
        print(f'\n🤖 {name.split("/")[-1]}: {hyps[idx][:200]}')


## 6. Exportando Resultados (Colab)
Compacta a planilha e os gráficos para download fácil.

In [ ]:
!zip -r resultados_avaliacao.zip resultados_avaliacao.csv avaliacao_metricas.png loss_por_epoca.png